# Conditional Tasks and Multimodal Agents [Step 5 -- Dynamic workflows and vision]

> **MLCourse - Agentic AI - CrewAI Advanced Agents**

Real-world workflows are not always linear. Sometimes task B should only run if
task A produced a specific result. CrewAI supports this with `ConditionalTask`
and `Rule`. This notebook also covers multimodal agents -- agents that process
images alongside text using vision-capable models and tools.

## What you will learn

- `ConditionalTask` with `Rule`: conditional task execution based on runtime results
- Branching task flows: the task graph changes dynamically based on intermediate output
- Multimodal agents: processing images with vision-capable LLMs
- VisionTool and DALL-E tool integration (guarded for optional dependencies)
- Building a crew that routes tasks based on input type

In [ ]:
# === SETUP CELL ===
import os
from pathlib import Path

from dotenv import load_dotenv

# Walk up from cwd until we reach the track root folder "03_agentic_ai".
# This lets the notebook run from any subfolder while finding the shared .env.
TRACK = Path.cwd()
while TRACK.name != "03_agentic_ai" and TRACK != TRACK.parent:
    TRACK = TRACK.parent
load_dotenv(TRACK / ".env")

# Guard Jupyter-only magic so this file stays valid as plain Python too.
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

print("Setup complete. Track root resolved to:", TRACK)

## 1. `ConditionalTask` and `Rule` -- branching execution

A `ConditionalTask` is a task that only runs if its condition evaluates to True.
The condition is defined by a `Rule`, which takes a function that receives the
output of a previous task and returns True/False.

**How it works:**
1. A prerequisite task runs normally and produces output.
2. The `Rule` function receives that output as a string.
3. If the function returns True, the ConditionalTask executes.
4. If False, the task is skipped entirely.

This enables dynamic workflows: the same crew can take different execution
paths depending on intermediate results -- like an if/else branch in code.

In [ ]:
from crewai import Agent, Task, Crew, Process, ConditionalTask, Rule
from langchain_ollama import ChatOllama

# Create the LLM -- ChatOllama with llama3.1:8b, local and free.
llm = ChatOllama(model="llama3.1:8b", temperature=0)

# An agent that classifies input and routes to the right handler.
classifier_agent = Agent(
    role="Input Classifier",
    goal="Classify input text and route it to the appropriate handler.",
    backstory=(
        "You are a precise classifier. You read input text and determine "
        "whether it is a question, a complaint, or a general statement. "
        "You output exactly one word: question, complaint, or statement."
    ),
    llm=llm,
    verbose=False,
    allow_delegation=False,
)

# Agent that handles questions.
question_agent = Agent(
    role="Question Answerer",
    goal="Answer factual questions clearly and concisely.",
    backstory="You are a knowledgeable assistant who provides direct answers.",
    llm=llm,
    verbose=False,
    allow_delegation=False,
)

# Agent that handles complaints.
complaint_agent = Agent(
    role="Complaint Handler",
    goal="Acknowledge complaints empathetically and propose solutions.",
    backstory=(
        "You are a customer success specialist who turns complaints into "
        "positive outcomes by acknowledging issues and offering solutions."
    ),
    llm=llm,
    verbose=False,
    allow_delegation=False,
)

# Agent that handles general statements.
statement_agent = Agent(
    role="General Responder",
    goal="Provide a brief, relevant response to general statements.",
    backstory="You are a conversational agent who keeps interactions friendly.",
    llm=llm,
    verbose=False,
    allow_delegation=False,
)

print("4 agents created: classifier, question handler, complaint handler, general handler")

## 2. Define condition functions for routing

The condition functions are plain Python functions that receive the previous
task's output as a string and return True/False. The LLM's classification
output is parsed to determine which branch to take.

**Key insight:** the condition function receives the RAW output of the previous
task (a string), so it needs to parse that string to extract the classification.
Keeping condition functions simple and robust is critical -- if the parsing
fails, the fallback is usually to skip the task.

In [ ]:
# Condition: run question handler if the classification contains "question".
def is_question(classification: str) -> bool:
    """Return True if the classification indicates a question."""
    # Lowercase and check for the keyword -- handles variations in LLM output.
    return "question" in classification.lower()


# Condition: run complaint handler if the classification contains "complaint".
def is_complaint(classification: str) -> bool:
    """Return True if the classification indicates a complaint."""
    return "complaint" in classification.lower()


# Condition: run general handler for anything else (statement or unknown).
def is_statement(classification: str) -> bool:
    """Return True if the classification indicates a general statement."""
    text = classification.lower()
    # True if it is a statement, or if it is neither a question nor a complaint.
    return "statement" in text or ("question" not in text and "complaint" not in text)


# Test the condition functions with sample classification outputs.
test_cases = [
    "question",
    "This is a question",
    "complaint",
    "I have a complaint about your service",
    "statement",
    "This is a general statement.",
    "something completely unexpected",
]

for tc in test_cases:
    q = is_question(tc)
    c = is_complaint(tc)
    s = is_statement(tc)
    print(f"  '{tc}' -> Q:{q} C:{c} S:{s}")

## 3. Build the conditional crew

The crew uses `Process.sequential` and defines three `ConditionalTask` entries
after the classification task. Each conditional task has a `Rule` that wraps
one of our condition functions. Only the task whose rule matches will execute.

In [ ]:
# Step 1: classify the input.
classify_task = Task(
    description=(
        "Classify the following customer message into exactly one category: "
        "question, complaint, or statement.\n\n"
        "Message: 'Why is my order taking so long to arrive? I placed it a week ago.'\n\n"
        "Output ONLY the category word."
    ),
    expected_output="A single word: question, complaint, or statement.",
    agent=classifier_agent,
)

# Step 2a: conditional -- answer if it is a question.
question_task = ConditionalTask(
    description=(
        "The input was classified as a question. Provide a helpful, factual "
        "answer to: 'Why is my order taking so long to arrive?'"
    ),
    expected_output="A clear, helpful answer addressing the customer's question.",
    agent=question_agent,
    # Rule: only run if the classification was "question".
    rule=Rule(
        condition=is_question,
    ),
)

# Step 2b: conditional -- handle if it is a complaint.
complaint_task = ConditionalTask(
    description=(
        "The input was classified as a complaint. Acknowledge the customer's "
        "frustration about their slow order and propose a concrete solution."
    ),
    expected_output="An empathetic acknowledgment with a specific resolution offer.",
    agent=complaint_agent,
    # Rule: only run if the classification was "complaint".
    rule=Rule(
        condition=is_complaint,
    ),
)

# Step 2c: conditional -- general response for statements or unknowns.
statement_task = ConditionalTask(
    description=(
        "The input was classified as a general statement. Provide a brief, "
        "friendly acknowledgment."
    ),
    expected_output="A brief, friendly response acknowledging the statement.",
    agent=statement_agent,
    # Rule: only run if neither question nor complaint matched.
    rule=Rule(
        condition=is_statement,
    ),
)

# Assemble the crew with conditional tasks.
conditional_crew = Crew(
    agents=[classifier_agent, question_agent, complaint_agent, statement_agent],
    tasks=[classify_task, question_task, complaint_task, statement_task],
    process=Process.sequential,
    verbose=False,
)

print("Conditional crew assembled with 1 classifier + 3 conditional branches")
print("Only one branch will execute based on the classification result")

## 4. Run the conditional crew

Execute the crew and observe that only ONE of the three conditional tasks
runs -- the one whose rule matches the classification output. The other two
are skipped entirely. This saves tokens and ensures the right handler processes
the input.

In [ ]:
print("=== Running Conditional Crew ===")
print("Input: 'Why is my order taking so long to arrive? I placed it a week ago.'")
print("Expected classification: question")
print("Expected branch: question_task\n")

try:
    result = conditional_crew.kickoff()
    print("=== Conditional Crew Result ===")
    print(result)
except Exception as e:
    print(f"[crew error] {e}")
    print("Ensure Ollama is running: ollama serve && ollama pull llama3.1:8b")

## 5. Test with a different input -- different branch

Run the same crew with a complaint input. The classifier should output
"complaint", triggering the complaint handler instead of the question handler.
This demonstrates the dynamic routing: same crew, different execution path.

In [ ]:
# Reconfigure the classification task with a complaint input.
complaint_input_task = Task(
    description=(
        "Classify the following customer message into exactly one category: "
        "question, complaint, or statement.\n\n"
        "Message: 'This is terrible! Your product broke after one day and "
        "nobody is helping me fix it. I want a refund immediately!'\n\n"
        "Output ONLY the category word."
    ),
    expected_output="A single word: question, complaint, or statement.",
    agent=classifier_agent,
)

complaint_crew = Crew(
    agents=[classifier_agent, question_agent, complaint_agent, statement_agent],
    tasks=[complaint_input_task, question_task, complaint_task, statement_task],
    process=Process.sequential,
    verbose=False,
)

print("=== Running Conditional Crew (complaint input) ===")
print("Input: 'This is terrible! Your product broke...'")
print("Expected classification: complaint")
print("Expected branch: complaint_task\n")

try:
    result = complaint_crew.kickoff()
    print("=== Complaint Branch Result ===")
    print(result)
except Exception as e:
    print(f"[crew error] {e}")

## 6. Multimodal agents -- processing images with vision models

CrewAI supports multimodal agents that can process images alongside text.
This requires a vision-capable LLM (like LLaVA via Ollama) and tools that
handle image inputs. The pattern is:

1. Use a vision-capable model: `ollama/llava` or `ollama/llama3.2-vision`
2. Pass images as part of the task description or via a vision tool
3. The agent processes both text and image content in its reasoning

**Note:** Vision models require more VRAM and may run slower than text-only
models. The examples below are guarded for environments without vision support.

In [ ]:
# Check if vision models are available via Ollama.
import subprocess

vision_model = None
try:
    result = subprocess.run(
        ["ollama", "list"],
        capture_output=True,
        text=True,
        timeout=5,
    )
    available = result.stdout.lower()
    if "llava" in available:
        vision_model = "llava"
    elif "llama3.1:8b-vision" in available:
        vision_model = "llama3.1:8b-vision"
    print(f"Vision model available: {vision_model or 'none detected'}")
    if not vision_model:
        print("To enable vision: ollama pull llava")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("[info] Ollama not reachable -- vision examples will be guarded")

## 7. VisionTool -- image analysis with CrewAI

CrewAI provides `VisionTool` (from `crewai_tools`) for analyzing images.
The tool accepts an image path or URL and returns a description generated
by the vision model. This is useful for agents that need to understand
visual content as part of their workflow.

**Important:** `VisionTool` requires a vision-capable model and the
`crewai_tools` package. This cell is guarded for environments without it.

In [ ]:
try:
    from crewai_tools import VisionTool

    # Create the vision tool -- it wraps an LLM call for image analysis.
    vision_tool = VisionTool()

    print("VisionTool imported successfully")
    print(f"Tool name: {vision_tool.name}")
    print("Usage: vision_tool.run(path_to_image)")
    print("The tool sends the image to a vision model and returns a text description")

    # If a vision model is available, create a multimodal agent.
    if vision_model:
        vision_agent = Agent(
            role="Image Analyst",
            goal="Analyze images and provide detailed descriptions.",
            backstory="You are an expert at understanding visual content.",
            tools=[vision_tool],
            llm=f"ollama/{vision_model}",
            verbose=False,
        )
        print(f"Multimodal agent created with model: {vision_model}")
    else:
        print("[skip] No vision model available -- agent creation skipped")
        print("Install one: ollama pull llava")

except ImportError:
    print("[skip] crewai_tools.VisionTool not available")
    print("Install: pip install crewai-tools")

## 8. DALL-E image generation tool (guarded)

CrewAI can also generate images using DALL-E or similar models. This requires
an OpenAI API key and the `dall-e` model access. The tool is guarded because
it requires a paid API key that may not be configured.

**Pattern for optional tools:** always guard tool imports and creation behind
environment variable checks. This keeps the notebook self-contained and
executable without API keys.

In [ ]:
# Guard the DALL-E tool behind an OpenAI API key check.
openai_key = os.getenv("OPENAI_API_KEY")

if openai_key:
    try:
        from crewai_tools import Dall-ETool

        # Create the DALL-E tool for image generation.
        dalle_tool = Dall-ETool()

        print("DALL-E tool imported successfully")
        print(f"Tool name: {dalle_tool.name}")
        print("Usage: dalle_tool.run('a futuristic city skyline at sunset')")
        print("Requires OPENAI_API_KEY in your .env file")

        # Create an agent that can generate images.
        image_agent = Agent(
            role="Image Generator",
            goal="Generate images based on text descriptions.",
            backstory="You are a creative artist who translates ideas into visuals.",
            tools=[dalle_tool],
            llm="ollama/llama3.1:8b",
            verbose=False,
        )
        print("Image generation agent created")

    except ImportError:
        print("[skip] crewai_tools.Dall-ETool not available")
        print("Install: pip install crewai-tools")
    except Exception as e:
        print(f"[skip] DALL-E tool setup failed: {e}")
else:
    print("[skip] No OPENAI_API_KEY found -- DALL-E tool requires a paid API key")
    print("To enable: add OPENAI_API_KEY=sk-... to your .env file")
    print("Note: DALL-E usage incurs OpenAI API costs")

## 9. Building a multimodal routing crew

Combine conditional tasks with multimodal capabilities to build a crew that
routes based on INPUT TYPE: text-only queries go to a text agent, while
image-related queries go to a vision agent. This is a common production
pattern where the same entry point handles diverse input types.

In [ ]:
# Text analysis agent -- handles pure text queries.
text_analyst = Agent(
    role="Text Analyst",
    goal="Analyze and answer text-based queries accurately.",
    backstory="You are a precise text analyst with strong reasoning skills.",
    llm=llm,
    verbose=False,
    allow_delegation=False,
)

# Vision analysis agent -- handles image-related queries.
# Uses the standard text model as a fallback when no vision model is available.
vision_analyst = Agent(
    role="Vision Analyst",
    goal="Analyze images and visual content, providing detailed descriptions.",
    backstory=(
        "You are an expert at understanding visual content. You describe images "
        "in detail and answer questions about visual information."
    ),
    llm=f"ollama/{vision_model}" if vision_model else llm,
    verbose=False,
    allow_delegation=False,
)

# Step 1: determine if the input involves an image.
type_detector_task = Task(
    description=(
        "Determine if the following request involves an image or visual content. "
        "Output exactly one word: text or visual.\n\n"
        "Request: 'Describe what is shown in this screenshot of a dashboard.'"
    ),
    expected_output="A single word: text or visual.",
    agent=classifier_agent,
)

# Conditional: text analysis for text-only inputs.
text_branch = ConditionalTask(
    description=(
        "Analyze the text-only request and provide a detailed response."
    ),
    expected_output="A detailed text analysis or answer.",
    agent=text_analyst,
    rule=Rule(condition=lambda x: "text" in x.lower()),
)

# Conditional: vision analysis for image-related inputs.
vision_branch = ConditionalTask(
    description=(
        "Analyze the visual content described in the request. "
        "Provide a detailed description of what the image likely shows."
    ),
    expected_output="A detailed visual analysis or description.",
    agent=vision_analyst,
    rule=Rule(condition=lambda x: "visual" in x.lower()),
)

# Assemble the multimodal routing crew.
multimodal_crew = Crew(
    agents=[classifier_agent, text_analyst, vision_analyst],
    tasks=[type_detector_task, text_branch, vision_branch],
    process=Process.sequential,
    verbose=False,
)

print("Multimodal routing crew assembled")
print("Routes to text_branch or vision_branch based on input type detection")

## 10. Run the multimodal routing crew

Execute the crew and observe the routing. The classifier determines the input
type, and the appropriate branch executes. For a visual input, the vision
analyst handles it; for text, the text analyst takes over.

In [ ]:
print("=== Running Multimodal Routing Crew ===")
print("Request: 'Describe what is shown in this screenshot of a dashboard.'")
print("Expected routing: visual -> vision_branch\n")

try:
    result = multimodal_crew.kickoff()
    print("=== Multimodal Crew Result ===")
    print(result)
except Exception as e:
    print(f"[crew error] {e}")
    print("Ensure Ollama is running: ollama serve && ollama pull llama3.1:8b")

## Summary and key takeaways

- `ConditionalTask` with `Rule` enables dynamic branching: task execution
  depends on the output of previous tasks.
- Condition functions are plain Python functions that receive a string output
  and return True/False -- keep them simple and robust.
- Multimodal agents process images via vision-capable models (llava, etc.)
  and tools like `VisionTool`.
- `Dall-ETool` generates images but requires an OpenAI API key (guarded).
- Always guard optional dependencies behind environment checks so notebooks
  remain self-contained and executable without API keys.
- Combining conditional routing with multimodal capabilities creates flexible
  entry points that handle diverse input types with the same crew.

**This completes the CrewAI Advanced Agents series.** You now have tools for
building custom tools, knowledge-grounded agents, persistent memory, reasoning,
and dynamic conditional workflows.